<a href="https://colab.research.google.com/github/toche7/BasicAIAgentColab/blob/main/BasicHarness3_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent: Agentic Loop

### ส่วนที่ 1: การกำหนดเครื่องมือ (Tool Definitions)

โค้ดส่วนนี้ทำหน้าที่กำหนดเครื่องมือ (Tools) ที่เอเจนต์ของเราสามารถเลือกใช้ได้ ได้แก่:
1. `calc`: ฟังก์ชันสำหรับคำนวณทางคณิตศาสตร์อย่างปลอดภัย โดยจำกัดให้ใส่ได้เฉพาะตัวเลขและเครื่องหมายพื้นฐานเท่านั้น
2. `read_file`: ฟังก์ชันสำหรับอ่านไฟล์ในไดเรกทอรีทำงานปัจจุบันอย่างปลอดภัย

พร้อมทั้งระบุโครงสร้าง (Schemas) เพื่อส่งต่อให้โมเดลเข้าใจหน้าที่และรูปแบบการรับส่งพารามิเตอร์ของแต่ละเครื่องมือ

In [1]:
"""  Part 1 - Tool Definitions"""
import json  # Used for parsing and formatting JSON payloads for tool arguments
import re  # Used for secure input validation of arithmetic expressions
from pathlib import Path  # Used for safe, cross-platform file path resolution

def calc(expression: str) -> str:
    """Evaluate an arithmetic expression securely by limiting allowed characters."""
    # Only allow numbers, spaces, parentheses, and standard arithmetic operators
    if not re.match(r'^[0-9+\-*/().\s]+$', expression):
        raise ValueError("only numbers and + - * / are allowed")
    try:
        # Safe to evaluate since we restricted character set completely
        return str(eval(expression, {"__builtins__": None}, {}))
    except Exception as e:
        raise ValueError(f"Invalid math expression: {e}")

def read_file(path: str) -> str:
    """Read a local file safely within the working directory."""
    p = Path(path).resolve()
    if not p.is_relative_to(Path.cwd().resolve()):
        raise PermissionError("path is outside the working directory")
    return p.read_text()[:8000]


# Tool schemas to pass to the model
SCHEMAS = [
    {"type": "function", "function": {
        "name": "calc", "description": "Evaluate an arithmetic expression like '12 * (3 + 4)'.",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}},
                       "required": ["expression"]}}},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a text file in the working directory.",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}},
                       "required": ["path"]}}},
]


### การแมปและการทำงานของเครื่องมือ (Tool Mapping & Execution)

ในส่วนนี้เป็นการรวบรวมฟังก์ชันทั้งหมดที่กำหนดไว้ก่อนหน้าเข้าไปในพจนานุกรม (Dictionary) ชื่อ `FUNCTIONS` และสร้างฟังก์ชัน `execute` เพื่อเรียกใช้เครื่องมือเหล่านั้นตามชื่อและพารามิเตอร์ที่โมเดลส่งกลับมาในรูปแบบ JSON

In [2]:

# Mapping of tool names to their respective functions
FUNCTIONS = {"calc": calc, "read_file": read_file}


def execute(name: str, arguments: str) -> str:
    """Execute a specified tool with arguments."""
    try:
        return FUNCTIONS[name](**json.loads(arguments or "{}"))
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"

### ส่วนที่ 2: การแชทและการเชื่อมต่อกับโมเดล (Chat & Runtime execution)

โค้ดส่วนนี้เป็นการดึง API Key ของ Groq เพื่อเริ่มต้นการใช้งาน และสร้างฟังก์ชัน `chat` สำหรับส่งประวัติการสนทนา (Conversation History) และเครื่องมือ (Tools) ไปยัง Groq API โดยใช้โมเดล `qwen3.8-27b` ผ่านไลบรารี `httpx`

In [3]:
"""Step 2 (Split): Part 2 - Chat and Runtime execution"""
import os
import sys
import httpx
from google.colab import userdata


URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL = "qwen/qwen3.8-27b"

try:
    api_key = userdata.get('GROQ_API_KEY')
except Exception:
    api_key = os.environ.get('GROQ_API_KEY')

def chat(messages: list[dict], tools: list[dict] | None = None) -> dict:
    """POST the conversation with tools configuration to Groq API."""
    if not api_key:
        raise ValueError("GROQ_API_KEY not found.")

    body = {"model": MODEL, "messages": messages, "max_tokens": 800}
    if tools:
        body["tools"] = tools

    headers = {"Authorization": f"Bearer {api_key}"}
    resp = httpx.post(URL, headers=headers, json=body, timeout=60)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]


### ส่วนที่ 3: ลูปของเอเจนต์ (The Agentic Loop)

ฟังก์ชัน `run_agent` เป็นหัวใจหลักของ Agentic Loop ซึ่งจะทำหน้าที่:
1. ส่งประวัติการสนทนาไปให้โมเดลประเมิน
2. เช็คว่าโมเดลต้องการเรียกใช้งานเครื่องมือ (Tool Calls) หรือไม่
3. ถ้าต้องการเรียกใช้งานเครื่องมือ ตัวเอเจนต์จะรันฟังก์ชันนั้น ดึงผลลัพธ์ใส่ประวัติการสนทนา และถามโมเดลซ้ำ
4. ทำซ้ำจนกว่าโมเดลจะได้คำตอบสุดท้ายโดยไม่ต้องใช้เครื่องมือเพิ่ม

In [4]:
"""Step 3 (new file): the agent loop. Call the model, run requested tools, repeat until it answers."""
def run_agent(messages: list[dict], max_steps: int = 8) -> str:
    for _ in range(max_steps):  # max_steps prevents infinite loops
        reply = chat(messages, SCHEMAS)
        calls = reply.get("tool_calls") or []
        messages.append({"role": "assistant", "content": reply.get("content") or "",
                         **({"tool_calls": calls} if calls else {})})
        if not calls:
            return messages[-1]["content"]
        for call in calls:
            fn = call["function"]
            print(f"[tool] {fn['name']}({fn['arguments']})")
            messages.append({"role": "tool", "tool_call_id": call["id"],
                             "content": execute(fn["name"], fn["arguments"])})
    return f"[stopped after {max_steps} steps without a final answer]"

In [5]:
question =  "hello how are you?"
print(run_agent([{"role": "user", "content": question}]))

I'm doing well, thank you! How can I help you today?


In [6]:
question =  "123+68"
print(run_agent([{"role": "user", "content": question}]))

[tool] calc({"expression":"123+68"})
123 + 68 = **191**


In [7]:
question =  "what inside note.txt"
print(run_agent([{"role": "user", "content": question}]))

[tool] read_file({"path":"note.txt"})
There's no file named `note.txt` in the working directory. It may not exist yet, or it might be under a different name or path. Could you double-check the filename or location?


### คลาสเซสชันการแชท (ChatSession Class)

คลาส `ChatSession` ทำหน้าที่คอยจำประวัติการสนทนาทั้งหมดไว้ในตัวแปร `self.messages` เพื่อให้ผู้ใช้สามารถสนทนาตอบโต้กับเอเจนต์ได้อย่างต่อเนื่อง (Multi-turn Conversation)

In [8]:
class ChatSession:
    def __init__(self):
        # Keep track of the full conversation history
        self.messages = []

    def send_message(self, user_message: str) -> str:
        # 1. Append the new user query to the history
        self.messages.append({"role": "user", "content": user_message})

        # 2. Run the agent loop starting with the current history
        # Note: run_agent will append assistant replies and tool results to self.messages in place
        response = run_agent(self.messages)
        return response

### ส่วนต่อประสานผู้ใช้แบบโต้ตอบ (Interactive Chat Widget UI)

โค้ดส่วนนี้ใช้ `ipywidgets` ในการสร้างหน้าต่างแชทจำลองที่สวยงามภายใน Jupyter Notebook เพื่อให้คุณสามารถพิมพ์คุยกับเอเจนต์และดูขั้นตอนการทำงาน (เช่น การเรียกใช้เครื่องมือ) ได้แบบเรียลไทม์

In [9]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Initialize a fresh chat session
ui_session = ChatSession()

# HTML template for chat messages
def format_message(sender, message):
    bg_color = "#E3F2FD" if sender == "You" else "#FFEBEE" if sender == "System Error" else "#F5F5F5"
    align = "right" if sender == "You" else "left"
    margin = "margin-left: 20%;" if sender == "You" else "margin-right: 20%;"
    text_color = "color: #C62828;" if sender == "System Error" else ""
    return f'''
    <div style="text-align: {align}; margin: 5px 0;">
        <div style="display: inline-block; background-color: {bg_color}; padding: 8px 12px;
                    border-radius: 12px; max-width: 80%; text-align: left; {margin} {text_color}">
            <strong>{sender}:</strong> {message.replace('\n', '<br>')}
        </div>
    </div>
    '''

# UI Elements - Fixed overflow property for correct scrolling
output_area = widgets.Output(layout=widgets.Layout(height='300px', overflow='auto', border='1px solid #ccc', padding='10px'))
text_input = widgets.Text(placeholder='Type your message here...', layout=widgets.Layout(width='80%'))
send_button = widgets.Button(description='Send', button_style='primary', layout=widgets.Layout(width='18%'))

# Action on sending a message
def on_send_click(b):
    user_msg = text_input.value.strip()
    if not user_msg:
        return

    # Clear input immediately for better UX
    text_input.value = ''

    # Append user message to UI
    with output_area:
        display(HTML(format_message("You", user_msg)))

    # Get agent response safely
    try:
        agent_reply = ui_session.send_message(user_msg)
        # Append agent reply to UI
        with output_area:
            display(HTML(format_message("Agent", agent_reply)))
    except Exception as e:
        with output_area:
            error_msg = str(e)
            if "429" in error_msg:
                error_msg = "Rate limit reached (HTTP 429). Please wait a brief moment and try again."
            display(HTML(format_message("System Error", error_msg)))

send_button.on_click(on_send_click)
text_input.on_submit(on_send_click) # Send when pressing Enter

# Display the Layout
print("Interactive Chat Widget:")
display(widgets.VBox([output_area, widgets.HBox([text_input, send_button])]))

Interactive Chat Widget:


[tool] calc({"expression":"123*9"})
[tool] calc({"expression":"258/4"})
[tool] read_file({"path":"note.tx"})
[tool] read_file({"path":"note.txt"})
